<a href="https://colab.research.google.com/github/prem242892-prog/GPS-TRILATERATION/blob/main/GPSipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ================================================================
# GPS TRILATERATION SIMULATOR (A6)
# B.Tech Mathematics Mini Project
# ================================================================
#
# Mathematical Concept:
# System of Linear Equations from Nonlinear Circle Equations
#
# Assumption:
# We use a 2D Cartesian coordinate system.
# Each tower has a known (x, y) position and a measured distance
# from the phone.
# ================================================================

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# ================================================================
# 1. TITLE
# ================================================================

print("=" * 70)
print("              GPS TRILATERATION SIMULATOR (A6)")
print("=" * 70)
print("Find the phone location using 3 cell towers.")
print()

# ================================================================
# 2. USER INPUT
# ================================================================

print("ENTER TOWER COORDINATES")
print("-" * 40)

x1 = float(input("Tower 1 - X coordinate: "))
y1 = float(input("Tower 1 - Y coordinate: "))

x2 = float(input("Tower 2 - X coordinate: "))
y2 = float(input("Tower 2 - Y coordinate: "))

x3 = float(input("Tower 3 - X coordinate: "))
y3 = float(input("Tower 3 - Y coordinate: "))

print()
print("ENTER DISTANCES FROM PHONE TO EACH TOWER")
print("-" * 40)

d1 = float(input("Distance from Tower 1: "))
d2 = float(input("Distance from Tower 2: "))
d3 = float(input("Distance from Tower 3: "))

# Store tower data
towers = np.array([
    [x1, y1],
    [x2, y2],
    [x3, y3]
], dtype=float)

distances = np.array([d1, d2, d3], dtype=float)

# ================================================================
# 3. INPUT VALIDATION
# ================================================================

if d1 <= 0 or d2 <= 0 or d3 <= 0:
    raise ValueError("ERROR: All distances must be greater than zero.")

if len(set(map(tuple, towers))) < 3:
    raise ValueError("ERROR: Tower coordinates must be different.")

# Check whether towers are collinear
area = (
    x1 * (y2 - y3)
    + x2 * (y3 - y1)
    + x3 * (y1 - y2)
)

if abs(area) < 1e-10:
    raise ValueError(
        "ERROR: The three towers are collinear. "
        "Please enter three non-collinear tower positions."
    )

print("\nInput validation successful.")

# ================================================================
# 4. MATHEMATICAL MODEL
# ================================================================

print("\n" + "=" * 70)
print("STEP 1: CIRCLE EQUATIONS")
print("=" * 70)

print("""
For every tower:

(x - xi)^2 + (y - yi)^2 = di^2

Each tower therefore creates one circle around the tower.
The phone location is the common intersection of these circles.
""")

print("Tower 1:")
print(f"(x - {x1})^2 + (y - {y1})^2 = {d1}^2")

print("\nTower 2:")
print(f"(x - {x2})^2 + (y - {y2})^2 = {d2}^2")

print("\nTower 3:")
print(f"(x - {x3})^2 + (y - {y3})^2 = {d3}^2")

# ================================================================
# 5. SUBTRACT EQUATIONS
# ================================================================

print("\n" + "=" * 70)
print("STEP 2: CONVERT TO LINEAR EQUATIONS")
print("=" * 70)

print("""
Expanding the circle equations gives x^2 and y^2 terms.

When Equation 1 is subtracted from Equation 2,
and Equation 1 is subtracted from Equation 3,

the common x^2 + y^2 terms cancel.

This produces two linear equations:

A[x, y]^T = b
""")

# Linear coefficient matrix
A = np.array([
    [2 * (x2 - x1), 2 * (y2 - y1)],
    [2 * (x3 - x1), 2 * (y3 - y1)]
], dtype=float)

# Right-hand side
b = np.array([
    d1**2 - d2**2
    + x2**2 + y2**2
    - x1**2 - y1**2,

    d1**2 - d3**2
    + x3**2 + y3**2
    - x1**2 - y1**2
], dtype=float)

print("Coefficient Matrix A:")
print(A)

print("\nRight-hand Side Vector b:")
print(b)

print("\nLinear Equations:")

print(
    f"{A[0,0]:.4f}x + "
    f"{A[0,1]:.4f}y = "
    f"{b[0]:.4f}"
)

print(
    f"{A[1,0]:.4f}x + "
    f"{A[1,1]:.4f}y = "
    f"{b[1]:.4f}"
)

# ================================================================
# 6. SOLVE THE LINEAR SYSTEM
# ================================================================

print("\n" + "=" * 70)
print("STEP 3: SOLVE THE SYSTEM")
print("=" * 70)

try:
    solution = np.linalg.solve(A, b)
except np.linalg.LinAlgError:
    raise ValueError("The system does not have a unique solution.")

phone_x = solution[0]
phone_y = solution[1]

print("\nEstimated Phone Location:")
print(f"X = {phone_x:.6f}")
print(f"Y = {phone_y:.6f}")

# ================================================================
# 7. VERIFY RESULT
# ================================================================

print("\n" + "=" * 70)
print("STEP 4: VERIFY THE RESULT")
print("=" * 70)

# Calculate distances from estimated phone position
calculated_distances = np.sqrt(
    (phone_x - towers[:, 0])**2 +
    (phone_y - towers[:, 1])**2
)

errors = calculated_distances - distances

print()
print(
    f"{'Tower':<10}"
    f"{'Input Distance':<20}"
    f"{'Calculated Distance':<23}"
    f"{'Error':<15}"
)

print("-" * 68)

for i in range(3):
    print(
        f"{i+1:<10}"
        f"{distances[i]:<20.4f}"
        f"{calculated_distances[i]:<23.4f}"
        f"{errors[i]:<15.6f}"
    )

max_error = np.max(np.abs(errors))

print("\nMaximum Absolute Error:", f"{max_error:.6f}")

# ================================================================
# 8. FINAL RESULT
# ================================================================

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(
    f"""
The estimated phone location is:

             ({phone_x:.4f}, {phone_y:.4f})

X Coordinate : {phone_x:.4f}
Y Coordinate : {phone_y:.4f}
"""
)

# ================================================================
# 9. VISUALIZATION - THREE CIRCLES
# ================================================================

fig, ax = plt.subplots(figsize=(10, 8))

theta = np.linspace(0, 2 * np.pi, 500)

for i, ((tx, ty), distance) in enumerate(
    zip(towers, distances)
):

    circle_x = tx + distance * np.cos(theta)
    circle_y = ty + distance * np.sin(theta)

    ax.plot(
        circle_x,
        circle_y,
        linewidth=2,
        label=f"Tower {i+1} Circle"
    )

    # Plot tower
    ax.scatter(
        tx,
        ty,
        marker="^",
        s=120,
        label=f"Tower {i+1}"
    )

    ax.text(
        tx,
        ty,
        f"  T{i+1}",
        fontsize=12,
        fontweight="bold"
    )

# Plot phone
ax.scatter(
    phone_x,
    phone_y,
    marker="*",
    s=300,
    label="Phone Location",
    zorder=10
)

ax.text(
    phone_x,
    phone_y,
    f"  Phone\n  ({phone_x:.2f}, {phone_y:.2f})",
    fontsize=11,
    fontweight="bold"
)

ax.set_xlabel("X Coordinate", fontsize=12)
ax.set_ylabel("Y Coordinate", fontsize=12)

ax.set_title(
    "GPS Trilateration - Phone Location",
    fontsize=16,
    fontweight="bold"
)

ax.grid(True, alpha=0.3)
ax.axis("equal")
ax.legend()

plt.tight_layout()
plt.show()

# ================================================================
# 10. DISTANCE COMPARISON CHART
# ================================================================

plt.figure(figsize=(9, 5))

tower_numbers = np.arange(3)
width = 0.35

plt.bar(
    tower_numbers - width / 2,
    distances,
    width,
    label="Input Distance"
)

plt.bar(
    tower_numbers + width / 2,
    calculated_distances,
    width,
    label="Calculated Distance"
)

plt.xticks(
    tower_numbers,
    ["Tower 1", "Tower 2", "Tower 3"]
)

plt.xlabel("Cell Towers")
plt.ylabel("Distance")

plt.title(
    "Input Distance vs Calculated Distance",
    fontsize=14,
    fontweight="bold"
)

plt.grid(axis="y", alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

# ================================================================
# 11. SYMBOLIC MATHEMATICAL MODEL USING SYMPY
# ================================================================

print("\n" + "=" * 70)
print("SYMBOLIC MATHEMATICAL MODEL")
print("=" * 70)

x, y = sp.symbols("x y")

eq1 = (x - x1)**2 + (y - y1)**2 - d1**2
eq2 = (x - x2)**2 + (y - y2)**2 - d2**2
eq3 = (x - x3)**2 + (y - y3)**2 - d3**2

print("\nExpanded Equation 1:")
sp.pprint(sp.expand(eq1))

print("\nExpanded Equation 2:")
sp.pprint(sp.expand(eq2))

print("\nExpanded Equation 3:")
sp.pprint(sp.expand(eq3))

# ================================================================
# 12. PROJECT SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)

print("""
GPS Trilateration Simulator

Input:
    - Coordinates of 3 cell towers
    - Distance from phone to each tower

Mathematical Process:
    1. Create 3 circle equations.
    2. Expand the equations.
    3. Subtract pairs of equations.
    4. Cancel x^2 + y^2.
    5. Obtain 2 linear equations.
    6. Solve the linear system.
    7. Obtain the estimated phone position.
    8. Verify the distances.
    9. Display the three circles and phone location.

Mathematical Concept:
    System of Linear Equations from Nonlinear Origins

Tools Used:
    Python
    NumPy
    SymPy
    Matplotlib
""")

print("=" * 70)
print("             SIMULATION COMPLETED SUCCESSFULLY")
print("=" * 70)

              GPS TRILATERATION SIMULATOR (A6)
Find the phone location using 3 cell towers.

ENTER TOWER COORDINATES
----------------------------------------
Tower 1 - X coordinate: 5
